In [ ]:
import rasterio
import numpy as np
import pandas as pd
import os

#Batch Processing of Monthly GeoTIFF Files Into Individual CSVs-----------------------

# === 1️⃣ Folder paths ===
input_folder = r"D:\ML ISI internship\Data\AET_GEOTIF_2015-25"
output_folder = r"D:\ML ISI internship\Data\AET_CSV_2015-25"

# Create output folder if it doesn’t exist
os.makedirs(output_folder, exist_ok=True)

# === 2️⃣ Loop through all GeoTIFF files ===
for file in os.listdir(input_folder):
    if file.lower().endswith(".tif"):
        file_path = os.path.join(input_folder, file)

        # Extract the last 6 digits before ".tif" (YYYYMM)
        base_name = os.path.splitext(file)[0]
        six_digits = base_name[-6:]   # e.g. "AET_SNPP_N_EB_MN_202409" → "202409"

        print(f"🛰 Processing {file} → Output: {six_digits}.csv")

        # === 3️⃣ Open GeoTIFF ===
        with rasterio.open(file_path) as src:
            band = src.read(1)
            transform = src.transform
            nodata = src.nodata

        # === 4️⃣ Mask invalid pixels ===
        mask = (band != 65535) & (band != 65534) & (~np.isnan(band))
        rows, cols = np.where(mask)

        # === 5️⃣ Convert to geographic coordinates ===
        lons, lats = rasterio.transform.xy(transform, rows, cols)
        values = band[rows, cols]

        # === 6️⃣ Round coordinates ===
        lats = np.round(lats, 2)
        lons = np.round(lons, 2)

        # === 7️⃣ Create DataFrame ===
        df = pd.DataFrame({
            "Latitude": lats,
            "Longitude": lons,
            "AET_value": values
        })

        # === 8️⃣ Remove duplicates after rounding (optional but recommended) ===
        df = df.groupby(["Latitude", "Longitude"], as_index=False)["AET_value"].mean()

        # === 9️⃣ Save output CSV (only six digits as filename) ===
        output_csv = os.path.join(output_folder, f"{six_digits}.csv")
        df.to_csv(output_csv, index=False)

        print(f"✅ Saved: {six_digits}.csv  ({len(df)} records)")

print("\n🎉 All GeoTIFF files converted successfully!")


In [6]:
#Cleaning the Reference Location CSV--------------------------------------------------

import os
import pandas as pd

folder_path = r"D:\ML ISI internship\Data\New folder (2)"
location_folder = r"D:\ML ISI internship\Data\Reference location csv"

# Make sure output folder exists
os.makedirs(location_folder, exist_ok=True)

for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
        file_path = os.path.join(folder_path, filename)
        print("Processing:", file_path)
        df = pd.read_csv(file_path)

        # Keeping only necessary columns
        df = df[["State Name", "District Name", "Latitude", "Longitude"]]

        # Round off Latitude and Longitude to 2 decimal places ===
        df['Latitude'] = df['Latitude'].round(2)
        df['Longitude'] = df['Longitude'].round(2)

        # Convert them to strings with exactly 2 decimal places ===
        #df['Latitude'] = df['Latitude'].map(lambda x: f"{x:.2f}")
        #df['Longitude'] = df['Longitude'].map(lambda x: f"{x:.2f}")

        # === 4️⃣ Sort alphabetically by 'State Name' and 'District Name' ===
        df_sorted = df.sort_values(by=['State Name', 'District Name'], ascending=[True, True])
        
        name_only, ext = os.path.splitext(filename)
        new_filename = name_only + "_cleaned" + ext
        
        # Full path for output file
        output_path = os.path.join(location_folder, new_filename)
        
        # Save CSV
        df_sorted.to_csv(output_path, index=False)
        
        print(f"Saved: {output_path}")

Processing: D:\ML ISI internship\Data\New folder (2)\ApportionedIdentifiers.csv
Saved: D:\ML ISI internship\Data\Reference location csv\ApportionedIdentifiers_cleaned.csv
Processing: D:\ML ISI internship\Data\New folder (2)\UnApportionedIdentifiers.csv
Saved: D:\ML ISI internship\Data\Reference location csv\UnApportionedIdentifiers_cleaned.csv


In [8]:
#Merging Monthly CSVs Into a Single Long-Format Dataset-------------------------------

import pandas as pd
import numpy as np
import os
from scipy.spatial import cKDTree

# === 1️⃣ Paths ===
location_folder = r"D:\ML ISI internship\Data\Reference location csv"
aet_folder = r"D:\ML ISI internship\Data\AET_CSV_2015-25"   # Folder containing AET files (e.g., 201503.csv → 202503.csv)
output_folder = r"D:\ML ISI internship\Data\Output folder"

# === 2️⃣ Load and clean location data ===
for filename in os.listdir(location_folder):
    if filename.endswith(".csv"):
        file_path = os.path.join(location_folder, filename)
        print("Processing:", file_path)
        locs = pd.read_csv(file_path)
        locs['Latitude'] = locs['Latitude'].round(2)
        locs['Longitude'] = locs['Longitude'].round(2)

        # === 3️⃣ Initialize final dataframe ===
        final_df = locs.copy()

        # === 4️⃣ Loop through all AET CSVs (sorted by date) ===
        aet_files = sorted([f for f in os.listdir(aet_folder) if f.endswith(".csv")])

        for file in aet_files:
            # Extract year and month from filename (e.g., "201503.csv" → 2015, 03)
            base_name = os.path.splitext(file)[0]
            year = base_name[:4]
            month = base_name[4:6]
        
            # Create the desired column header format → "01/MM/YYYY"
            month_id = f"01/{month}/{year}"
        
            print(f"🛰 Processing {file} → Column: {month_id}")
        
            # === 5️⃣ Load monthly AET data ===
            aet = pd.read_csv(os.path.join(aet_folder, file))
            aet['Latitude'] = aet['Latitude'].round(2)
            aet['Longitude'] = aet['Longitude'].round(2)
        
            # === 6️⃣ Exact match first ===
            merged = pd.merge(locs, aet, on=['Latitude', 'Longitude'], how='left')
        
            # === 7️⃣ Handle unmatched using nearest AET point ===
            unmatched = merged[merged['AET_value'].isna()].copy()
            if len(unmatched) > 0:
                print(f"   ⚠️ {len(unmatched)} unmatched → finding nearest points...")
        
                # Build KDTree from AET coordinates
                aet_coords = aet[['Latitude', 'Longitude']].to_numpy()
                tree = cKDTree(aet_coords)
        
                # Query nearest AET pixel for unmatched locations
                loc_coords = unmatched[['Latitude', 'Longitude']].to_numpy()
                distances, indices = tree.query(loc_coords, k=1)
        
                # Assign nearest AET values
                unmatched['AET_value'] = aet.iloc[indices]['AET_value'].values
                unmatched['distance_km'] = distances * 111  # degrees → km
        
                # Update main dataframe
                merged.update(unmatched)
            else:
                merged['distance_km'] = 0
        
            # === 8️⃣ Rename AET_value column to "01/MM/YYYY" ===
            merged.rename(columns={"AET_value": month_id}, inplace=True)
        
            # === 9️⃣ Keep only required columns ===
            merged = merged[['State Name', 'District Name', 'Latitude', 'Longitude', month_id]]
        
            # === 🔟 Merge with final dataset ===
            final_df = pd.merge(final_df, merged, on=['State Name', 'District Name', 'Latitude', 'Longitude'], how='left')

        print("\n✅ All AET files processed successfully!")

        #removing "Latitude" and "Longitude" columns from the final csv file
        final_df = final_df.drop(['Latitude', 'Longitude'], axis=1)

        # Convert all non-numeric or NaN cells to 0
        final_df.replace([np.inf, -np.inf], 0, inplace=True)
        final_df = final_df.applymap(lambda x: np.nan if isinstance(x, (float, np.floating)) and x < -1e+10 else x)
        
        for col in final_df.columns[2:]:  # Skip State, District
            final_df[col] = pd.to_numeric(final_df[col], errors='coerce').fillna(0)
        
        # === 5️⃣ Save final merged CSV ===
        name_only, ext = os.path.splitext(filename)
        new_filename = name_only + "_aet" + ext

        output_path = os.path.join(output_folder, new_filename)
        
        final_df.to_csv(output_path, index=False, float_format="%.3f")
        print(f"📁 Final dataset saved as: {output_path}")


Processing: D:\ML ISI internship\Data\Reference location csv\ApportionedIdentifiers_cleaned.csv
🛰 Processing 201503.csv → Column: 01/03/2015
   ⚠️ 9 unmatched → finding nearest points...
🛰 Processing 201504.csv → Column: 01/04/2015
   ⚠️ 9 unmatched → finding nearest points...
🛰 Processing 201505.csv → Column: 01/05/2015
   ⚠️ 8 unmatched → finding nearest points...
🛰 Processing 201506.csv → Column: 01/06/2015
   ⚠️ 13 unmatched → finding nearest points...
🛰 Processing 201507.csv → Column: 01/07/2015
   ⚠️ 12 unmatched → finding nearest points...
🛰 Processing 201508.csv → Column: 01/08/2015
   ⚠️ 13 unmatched → finding nearest points...
🛰 Processing 201509.csv → Column: 01/09/2015
   ⚠️ 8 unmatched → finding nearest points...
🛰 Processing 201510.csv → Column: 01/10/2015
   ⚠️ 9 unmatched → finding nearest points...
🛰 Processing 201511.csv → Column: 01/11/2015
   ⚠️ 8 unmatched → finding nearest points...
🛰 Processing 201512.csv → Column: 01/12/2015
   ⚠️ 7 unmatched → finding nearest p

In [14]:
#Transforming the Long-Format Dataset into Wide Format--------------------------------

import os
import pandas as pd

input_folder = r"D:\ML ISI internship\Data\Output folder"
output_folder = r"D:\ML ISI internship\Data\Output folder transposed"

# Make sure output folder exists
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):
        
        # Full path of input file
        input_path = os.path.join(input_folder, filename)
        
        # Read CSV
        df = pd.read_csv(input_path)
        
        # Transpose it
        df_T = df.transpose()
        
        # Create new filename by adding "_changed"
        name_only, ext = os.path.splitext(filename)
        new_filename = name_only + "_transposed" + ext
        
        # Full path for output file
        output_path = os.path.join(output_folder, new_filename)
        
        # Save CSV
        df_T.to_csv(output_path, header=False)
        
        print(f"Saved: {output_path}")


Saved: D:\ML ISI internship\Data\Output folder transposed\ApportionedIdentifiers_cleaned_aet_transposed.csv
Saved: D:\ML ISI internship\Data\Output folder transposed\UnApportionedIdentifiers_cleaned_aet_transposed.csv


In [15]:
import pandas as pd

df = pd.read_csv(r"D:\ML ISI internship\Data\Output folder transposed\ApportionedIdentifiers_cleaned_aet_transposed.csv")

print(df.shape)


(120, 314)


In [16]:
import pandas as pd

df = pd.read_csv(r"D:\ML ISI internship\Data\Output folder transposed\UnApportionedIdentifiers_cleaned_aet_transposed.csv")

print(df.shape)

(120, 603)


In [13]:
final_df=pd.read_csv(r"D:\ML ISI internship\Data\Output folder\UnApportionedIdentifiers_cleaned_aet.csv")
final_df.replace([np.inf, -np.inf], 0, inplace=True)
final_df = final_df.applymap(lambda x: np.nan if isinstance(x, (float, np.floating)) and x < -1e+10 else x)

# Convert all non-numeric or NaN cells to 0
for col in final_df.columns[2:]:  # Skip State, District
    final_df[col] = pd.to_numeric(final_df[col], errors='coerce').fillna(0)

final_df.to_csv("UnApportionedIdentifiers_cleaned_aet_1.csv", index=False, float_format="%.3f")

C:\Users\arkop\AppData\Local\Temp\ipykernel_33256\4216802443.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  final_df = final_df.applymap(lambda x: np.nan if isinstance(x, (float, np.floating)) and x < -1e+10 else x)


In [2]:
import os
import pandas as pd

folder_path = r"D:\ML ISI internship\Data\Reference location csv"

for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
        file_path = os.path.join(folder_path, filename)
        print("Processing:", file_path)
        df = pd.read_csv(file_path)
        print(df.head(10))
        

Processing: D:\ML ISI internship\Data\Reference location csv\ApportionedIdentifiers_filtered.csv
       State Name District Name  Latitude  Longitude
0    Chhattisgarh          Durg      21.2       81.3
1    Chhattisgarh        Bastar      19.1       82.0
2    Chhattisgarh        Raipur      21.2       81.7
3    Chhattisgarh      Bilaspur      22.1       82.1
4    Chhattisgarh       Raigarh      21.9       83.4
5    Chhattisgarh       Surguja      23.1       83.2
6  Madhya Pradesh      Jabalpur      23.0       80.0
7  Madhya Pradesh      Balaghat      22.1       80.6
8  Madhya Pradesh    Chhindwara      22.1       79.0
9  Madhya Pradesh   Narsinghpur      22.8       79.0
Processing: D:\ML ISI internship\Data\Reference location csv\UnApportionedIdentifiers_filtered.csv
       State Name District Name  Latitude  Longitude
0    Chhattisgarh          Durg      21.2       81.3
1    Chhattisgarh        Bastar      19.1       82.0
2    Chhattisgarh        Raipur      21.2       81.7
3    Chha

In [ ]:
import rasterio
import numpy as np
import pandas as pd
import os

#Batch Processing of Monthly GeoTIFF Files Into Individual CSVs-----------------------

# === 1️⃣ Folder paths ===
input_folder = r"D:\ML ISI internship\Data\AET_GEOTIF_2015-25"
output_folder = r"D:\ML ISI internship\Data\AET_CSV_2015-25"

# Create output folder if it doesn’t exist
os.makedirs(output_folder, exist_ok=True)

# === 2️⃣ Loop through all GeoTIFF files ===
for file in os.listdir(input_folder):
    if file.lower().endswith(".tif"):
        file_path = os.path.join(input_folder, file)

        # Extract the last 6 digits before ".tif" (YYYYMM)
        base_name = os.path.splitext(file)[0]
        six_digits = base_name[-6:]   # e.g. "AET_SNPP_N_EB_MN_202409" → "202409"

        print(f"🛰 Processing {file} → Output: {six_digits}.csv")

        # === 3️⃣ Open GeoTIFF ===
        with rasterio.open(file_path) as src:
            band = src.read(1)
            transform = src.transform
            nodata = src.nodata

        # === 4️⃣ Mask invalid pixels ===
        mask = (band != 65535) & (band != 65534) & (~np.isnan(band))
        rows, cols = np.where(mask)

        # === 5️⃣ Convert to geographic coordinates ===
        lons, lats = rasterio.transform.xy(transform, rows, cols)
        values = band[rows, cols]

        # === 6️⃣ Round coordinates ===
        lats = np.round(lats, 2)
        lons = np.round(lons, 2)

        # === 7️⃣ Create DataFrame ===
        df = pd.DataFrame({
            "Latitude": lats,
            "Longitude": lons,
            "AET_value": values
        })

        # === 8️⃣ Remove duplicates after rounding (optional but recommended) ===
        df = df.groupby(["Latitude", "Longitude"], as_index=False)["AET_value"].mean()

        # === 9️⃣ Save output CSV (only six digits as filename) ===
        output_csv = os.path.join(output_folder, f"{six_digits}.csv")
        df.to_csv(output_csv, index=False)

        print(f"✅ Saved: {six_digits}.csv  ({len(df)} records)")

print("\n🎉 All GeoTIFF files converted successfully!")


#Cleaning the Reference Location CSV--------------------------------------------------

folder_path = r"D:\ML ISI internship\Data\New folder"
location_folder = r"D:\ML ISI internship\Data\Reference location csv"

# Make sure output folder exists
os.makedirs(location_folder, exist_ok=True)

for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
        file_path = os.path.join(folder_path, filename)
        print("Processing:", file_path)
        df = pd.read_csv(file_path)

        # Keeping only necessary columns
        df = df[["State Name", "District Name", "Latitude", "Longitude"]]

        # Round off Latitude and Longitude to 2 decimal places ===
        df['Latitude'] = df['Latitude'].round(2)
        df['Longitude'] = df['Longitude'].round(2)

        # Convert them to strings with exactly 2 decimal places ===
        #df['Latitude'] = df['Latitude'].map(lambda x: f"{x:.2f}")
        #df['Longitude'] = df['Longitude'].map(lambda x: f"{x:.2f}")

        # === 4️⃣ Sort alphabetically by 'State Name' and 'District Name' ===
        df_sorted = df.sort_values(by=['State Name', 'District Name'], ascending=[True, True])
        
        name_only, ext = os.path.splitext(filename)
        new_filename = name_only + "_cleaned" + ext
        
        # Full path for output file
        output_path = os.path.join(location_folder, new_filename)
        
        # Save CSV
        df_sorted.to_csv(output_path, index=False)
        
        print(f"Saved: {output_path}")


#Merging Monthly CSVs Into a Single Long-Format Dataset-------------------------------

from scipy.spatial import cKDTree

# === 1️⃣ Paths ===
location_folder = r"D:\ML ISI internship\Data\Reference location csv"
aet_folder = r"D:\ML ISI internship\Data\AET_CSV_2015-25"   # Folder containing AET files (e.g., 201503.csv → 202503.csv)
output_folder = r"D:\ML ISI internship\Data\Output folder"

# === 2️⃣ Load and clean location data ===
for filename in os.listdir(location_folder):
    if filename.endswith(".csv"):
        file_path = os.path.join(location_folder, filename)
        print("Processing:", file_path)
        locs = pd.read_csv(file_path)
        locs['Latitude'] = locs['Latitude'].round(2)
        locs['Longitude'] = locs['Longitude'].round(2)

        # === 3️⃣ Initialize final dataframe ===
        final_df = locs.copy()

        # === 4️⃣ Loop through all AET CSVs (sorted by date) ===
        aet_files = sorted([f for f in os.listdir(aet_folder) if f.endswith(".csv")])

        for file in aet_files:
            # Extract year and month from filename (e.g., "201503.csv" → 2015, 03)
            base_name = os.path.splitext(file)[0]
            year = base_name[:4]
            month = base_name[4:6]
        
            # Create the desired column header format → "01/MM/YYYY"
            month_id = f"01/{month}/{year}"
        
            print(f"🛰 Processing {file} → Column: {month_id}")
        
            # === 5️⃣ Load monthly AET data ===
            aet = pd.read_csv(os.path.join(aet_folder, file))
            aet['Latitude'] = aet['Latitude'].round(2)
            aet['Longitude'] = aet['Longitude'].round(2)
        
            # === 6️⃣ Exact match first ===
            merged = pd.merge(locs, aet, on=['Latitude', 'Longitude'], how='left')
        
            # === 7️⃣ Handle unmatched using nearest AET point ===
            unmatched = merged[merged['AET_value'].isna()].copy()
            if len(unmatched) > 0:
                print(f"   ⚠️ {len(unmatched)} unmatched → finding nearest points...")
        
                # Build KDTree from AET coordinates
                aet_coords = aet[['Latitude', 'Longitude']].to_numpy()
                tree = cKDTree(aet_coords)
        
                # Query nearest AET pixel for unmatched locations
                loc_coords = unmatched[['Latitude', 'Longitude']].to_numpy()
                distances, indices = tree.query(loc_coords, k=1)
        
                # Assign nearest AET values
                unmatched['AET_value'] = aet.iloc[indices]['AET_value'].values
                unmatched['distance_km'] = distances * 111  # degrees → km
        
                # Update main dataframe
                merged.update(unmatched)
            else:
                merged['distance_km'] = 0
        
            # === 8️⃣ Rename AET_value column to "01/MM/YYYY" ===
            merged.rename(columns={"AET_value": month_id}, inplace=True)
        
            # === 9️⃣ Keep only required columns ===
            merged = merged[['State Name', 'District Name', 'Latitude', 'Longitude', month_id]]
        
            # === 🔟 Merge with final dataset ===
            final_df = pd.merge(final_df, merged, on=['State Name', 'District Name', 'Latitude', 'Longitude'], how='left')

        print("\n✅ All AET files processed successfully!")

        #removing "Latitude" and "Longitude" columns from the final csv file
        final_df = final_df.drop(['Latitude', 'Longitude'], axis=1)

        # Convert all non-numeric or NaN cells to 0
        final_df.replace([np.inf, -np.inf], 0, inplace=True)
        final_df = final_df.applymap(lambda x: np.nan if isinstance(x, (float, np.floating)) and x < -1e+10 else x)
        
        for col in final_df.columns[2:]:  # Skip State, District
            final_df[col] = pd.to_numeric(final_df[col], errors='coerce').fillna(0)
        
        # === 5️⃣ Save final merged CSV ===
        name_only, ext = os.path.splitext(filename)
        new_filename = name_only + "_aet" + ext

        output_path = os.path.join(output_folder, new_filename)
        
        final_df.to_csv(output_path, index=False, float_format="%.3f")
        print(f"📁 Final dataset saved as: {output_path}")


#Transforming the Long-Format Dataset into Wide Format--------------------------------

input_folder = r"D:\ML ISI internship\Data\Output folder"
output_folder = r"D:\ML ISI internship\Data\Output folder transposed"

# Make sure output folder exists
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):
        
        # Full path of input file
        input_path = os.path.join(input_folder, filename)
        
        # Read CSV
        df = pd.read_csv(input_path)
        
        # Transpose it
        df_T = df.transpose()
        
        # Create new filename by adding "_changed"
        name_only, ext = os.path.splitext(filename)
        new_filename = name_only + "_transposed" + ext
        
        # Full path for output file
        output_path = os.path.join(output_folder, new_filename)
        
        # Save CSV
        df_T.to_csv(output_path, header=False)
        
        print(f"Saved: {output_path}")
